In [25]:
from Bio import SeqIO
from subprocess import Popen, PIPE
from tqdm import tqdm
from joblib import Parallel, delayed

In [26]:
def read_fa(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id)
        seq = str(x.seq).replace("U", "T")
        res[id] = seq
    return res

def read_fa2(path):
    res = {}
    rescords = list(SeqIO.parse(path, format="fasta"))
    for x in rescords:
        id = str(x.id).split('|')[0]
        seq = str(x.seq) #.replace("U", "T")
        res[id] = seq
    return res

In [27]:
lnc_dict_seq = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/lncRNA.fa")

In [28]:
lnc_dict_ss = read_fa2("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_2d/lncRNA_2d_ss.fa")
lnc_list = list(lnc_dict_ss.keys())

In [29]:
def join_ss_seq(lnc):
  
  seq = lnc_dict_seq[lnc]
  ss = lnc_dict_ss[lnc]

  path = f"C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/ _{lnc}.txt"

  f = open(path, "w")

  f.write(seq + "\n")
  f.write(ss)

  f.close()

In [30]:
for lnc in tqdm(lnc_list):
  try:
    join_ss_seq(lnc)
  except:
    print(lnc)

100%|██████████| 50/50 [00:00<00:00, 41909.51it/s]

NONHSAT000043.2
NONHSAT000044.2
NONHSAT000173.2
NONHSAT000179.2
NONHSAT000201.2
NONHSAT000202.2
NONHSAT000203.2
NONHSAT000204.2
NONHSAT000228.2
NONHSAT000284.2
NONHSAT000530.2
NONHSAT000531.2
NONHSAT000532.2
NONHSAT000533.2
NONHSAT000534.2
NONHSAT000535.2
NONHSAT000536.2
NONHSAT000537.2
NONHSAT000538.2
NONHSAT000539.2
NONHSAT000540.2
NONHSAT000542.2
NONHSAT000543.2
NONHSAT000544.2
NONHSAT000545.2
NONHSAT000546.2
NONHSAT000834.2
NONHSAT000920.2
NONHSAT000923.2
NONHSAT001062.2
NONHSAT001116.2
NONHSAT001136.2
NONHSAT001159.2
NONHSAT001162.2
NONHSAT001209.2
NONHSAT001210.2
NONHSAT001211.2
NONHSAT001212.2
NONHSAT001258.2
NONHSAT001380.2
NONHSAT001460.2
NONHSAT001461.2
NONHSAT001462.2
NONHSAT001463.2
NONHSAT001464.2
NONHSAT001465.2
NONHSAT001466.2
NONHSAT001467.2
NONHSAT001468.2
NONHSAT001498.2


In [31]:
def eval_rna_structure(lnc):
  input = f'C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_{lnc}.txt'
  output = f'C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/ss_loops_lnc/loops_{lnc}.txt'
  
  command = f"wsl RNAeval -v -d2 < {input} > {output}"
  result = Popen(command, shell=True, stdout = PIPE)
  eval_loops, err = result.communicate() 

In [32]:
def worker(lnc):
    try:
        eval_rna_structure(lnc)
    except Exception as e:
        print(lnc)

if __name__ == '__main__':
    num_cores = 8
    results = Parallel(n_jobs=num_cores)(delayed(worker)(lnc) for lnc in tqdm(lnc_list))

  0%|          | 0/50 [00:00<?, ?it/s]/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000043.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000044.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000173.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000179.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000201.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000202.2.txt: No such file or directory
/bin/sh: C:/Users/steve/Documents/GitHub/CNN_lnc_miR/Datos/lnc/seq+ss_join_lnc/seq+ss_join_NONHSAT000203.2.txt: No such file or direct

In [33]:
import csv
import os

lnc_characterization_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Lnc/ss_loops_lnc/"
generated_file_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Lnc/lnc_features.csv"

files = os.listdir(lnc_characterization_path)

sequences = {}
for file_name in files:
    name = file_name[6:21]
    file_path = os.path.join(lnc_characterization_path, file_name)
    with open(file_path, 'r') as file:
        features = {"External": 0, "Interior": 0, "Hairpin": 0, "Multi": 0, "Energy": 0}
        lines = file.readlines()[:-2]
        for line in lines:
            tokens = line.split()
            features[tokens[0]] += 1
            features["Energy"] += int(tokens[-1])
        sequences[name] = features

sorted_sequences = dict(sorted(sequences.items()))
with open(generated_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    for seq in sorted_sequences.keys():
        features = sorted_sequences[seq]
        writer.writerow([seq, features["Interior"], features["Hairpin"], features["Multi"], float(features["Energy"]/100)])